# TxGNN KG schema/current status inspection

Safe-by-default notebook for inspecting schema lifecycle, canonical Parquet metadata, evidence files, and a bounded sample of one relation. It avoids full reads of large Parquets unless you explicitly raise `SAMPLE_ROWS`.


In [ ]:
from IPython.display import display
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

KG_ROOT = Path('/mnt/gcs/jouvencekb/kg/v2')
SAMPLE_RELATION = 'mutation_associated_disease'
SAMPLE_ROWS = 20  # keep bounded; do not set huge values in the notebook kernel
KG_ROOT


## Schema relations

Lifecycle/status comes from `manage_db.kg_schema`. Deprecated and legacy-index relations should not be promoted just to fill coverage percentages.


In [ ]:
from manage_db.kg_schema import RELATIONS

relations = pd.DataFrame([
    {
        'relation': r.name,
        'source': r.source.value,
        'target': r.target.value,
        'kind': r.kind.value,
        'direct': r.direct,
        'status': r.status.value,
        'replacement': r.replacement or '',
        'notes': r.notes,
    }
    for r in RELATIONS
])
relations['edge_file_exists'] = relations['relation'].map(lambda r: (KG_ROOT / 'edges' / f'{r}.parquet').exists())
relations.groupby(['status', 'edge_file_exists']).size().rename('n').reset_index()


In [ ]:
# Filter this dataframe interactively in Jupyter, e.g. by status/kind/missing edge file.
relations.sort_values(['status', 'kind', 'relation']).head(30)


## Canonical Parquet metadata only

This cell reads Parquet footers, not full data tables.


In [ ]:
def parquet_rows(path: Path) -> int | None:
    if not path.exists():
        return None
    return pq.ParquetFile(path).metadata.num_rows

node_files = sorted((KG_ROOT / 'nodes').glob('*.parquet'))
edge_files = sorted((KG_ROOT / 'edges').glob('*.parquet'))
evidence_files = sorted((KG_ROOT / 'evidence').glob('*.parquet')) if (KG_ROOT / 'evidence').exists() else []
summary = {
    'node_files': len(node_files),
    'edge_files': len(edge_files),
    'evidence_files': len(evidence_files),
    'node_rows': sum(parquet_rows(p) or 0 for p in node_files),
    'edge_rows': sum(parquet_rows(p) or 0 for p in edge_files),
    'evidence_rows': sum(parquet_rows(p) or 0 for p in evidence_files),
}
summary


In [ ]:
edge_meta = pd.DataFrame([{'relation': p.stem, 'rows': parquet_rows(p), 'path': str(p)} for p in edge_files])
evidence_meta = pd.DataFrame([{'relation': p.stem, 'evidence_rows': parquet_rows(p), 'evidence_path': str(p)} for p in evidence_files])
edge_status = relations.merge(edge_meta, on='relation', how='left').merge(evidence_meta, on='relation', how='left')
edge_status.sort_values(['edge_file_exists', 'rows', 'relation'], ascending=[True, False, True]).head(60)


## Phenotype direction sanity check

New exports should use entity → phenotype direction. Inverted/phenotype-indexed names are compatibility metadata only.


In [ ]:
edge_status[edge_status['relation'].str.contains('phenotype', regex=False)][['relation','source','target','status','replacement','edge_file_exists','rows','evidence_rows','notes']]


## Bounded relation sample

This reads at most `SAMPLE_ROWS` rows from one edge/evidence Parquet. Keep the value small.


In [ ]:
edge_path = KG_ROOT / 'edges' / f'{SAMPLE_RELATION}.parquet'
evidence_path = KG_ROOT / 'evidence' / f'{SAMPLE_RELATION}.parquet'
print('edge_path', edge_path, 'exists=', edge_path.exists())
print('evidence_path', evidence_path, 'exists=', evidence_path.exists())
if edge_path.exists():
    display(pq.read_table(edge_path).slice(0, SAMPLE_ROWS).to_pandas())
if evidence_path.exists():
    display(pq.read_table(evidence_path).slice(0, SAMPLE_ROWS).to_pandas())


## Active backlog view

Missing active relations are candidates for source review. Deprecated/legacy/derived relations are not automatic TODOs.


In [ ]:
active_missing = edge_status[(edge_status['status'] == 'active') & (~edge_status['edge_file_exists'])]
active_missing[['relation','source','target','kind','direct','notes']].sort_values('relation')
